# LLM Inference Baseline — Colab GPU run

End-to-end Phase 1: verify GPU, install deps, smoke-test the harness on a tiny model, then run the real long-context sweep, then summarize.

**Before starting:** `Runtime → Change runtime type → T4 GPU` (free) or A100 (Colab Pro+).


## 1. Confirm GPU

In [1]:
!nvidia-smi

Mon Apr 27 02:44:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Get the code

Edit `REPO_URL` to point at your GitHub repo, then run the cell. If the notebook is already inside a clone, the clone is skipped.

In [4]:
import os

REPO_URL = "https://github.com/sonavk2/LLM_Inference.git"  # <-- EDIT THIS

if not os.path.exists("scripts/run_context_sweep.py"):
    !git clone {REPO_URL}
    repo_name = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")
    %cd {repo_name}

!ls

Cloning into 'LLM_Inference'...
remote: Enumerating objects: 35, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 35 (delta 3), reused 34 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (35/35), 21.11 KiB | 3.52 MiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/LLM_Inference
CLAUDE.md  notebooks	       README.md	 results  src
configs    pyrightconfig.json  requirements.txt  scripts


## 3. Install Python deps

Colab already ships PyTorch with CUDA, so we only install the rest.

In [5]:
!pip install -q -r requirements.txt

## 4. (Optional) Hugging Face auth

Only needed for gated models like Llama-3. Accept the license on the model page first, then create a token at https://huggingface.co/settings/tokens and paste it into the prompt.

In [6]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## 5. Smoke test (~30s, tiny public model)

Verifies the harness end-to-end before pulling real weights. Should produce two JSONL rows in `results/baseline_hf.jsonl` with non-null TTFT and latency.

In [7]:
!python scripts/run_context_sweep.py \
  --config configs/baseline_hf.yaml \
  --model-id sshleifer/tiny-gpt2 \
  --dtype float16 \
  --context-lengths 128 512 1024 \
  --max-new-tokens 16

Loading sshleifer/tiny-gpt2 (dtype=float16) on cuda ...
config.json: 100% 662/662 [00:00<00:00, 3.89MB/s]
tokenizer_config.json: 100% 26.0/26.0 [00:00<00:00, 166kB/s]
vocab.json: 899kB [00:00, 10.5MB/s]
merges.txt: 456kB [00:00, 8.94MB/s]
special_tokens_map.json: 100% 90.0/90.0 [00:00<00:00, 567kB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
pytorch_model.bin: 100% 2.51M/2.51M [00:00<00:00, 4.02MB/s]
Loading weights: 100% 29/29 [00:00<00:00, 1969.19it/s, Materializing param=transformer.wte.weight]
The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: sshleifer/tiny-gpt2
Key                                   | Status     |  | 
--------------------------------------+------------+--+-
transformer.h.{0, 1}.attn.bias        | UNEXPECTED |  | 
tr

In [8]:
!cat results/baseline_hf.jsonl

{"model_name": "sshleifer/tiny-gpt2", "backend": "huggingface", "hardware": "cuda:Tesla T4", "context_length": 128, "batch_size": 1, "max_new_tokens": 16, "ttft_seconds": 0.9737673969998468, "tpot_seconds": 0.0723132401249984, "total_latency_seconds": 1.1570118419999744, "tokens_per_second": 13.828726223184459, "peak_gpu_memory_gb": 0.012604416, "kv_cache_memory_gb": 2.048e-06, "success": true, "error": null}
{"model_name": "sshleifer/tiny-gpt2", "backend": "huggingface", "hardware": "cuda:Tesla T4", "context_length": 512, "batch_size": 1, "max_new_tokens": 16, "ttft_seconds": 0.06006794399991122, "tpot_seconds": 0.007185914624997736, "total_latency_seconds": 0.11497463399996377, "tokens_per_second": 139.16113009766173, "peak_gpu_memory_gb": 0.017915392, "kv_cache_memory_gb": 8.192e-06, "success": true, "error": null}
{"model_name": "sshleifer/tiny-gpt2", "backend": "huggingface", "hardware": "cuda:Tesla T4", "context_length": 1024, "batch_size": 1, "max_new_tokens": 16, "ttft_seconds"

## 6. Real Phase 1 sweep

Edit the cell for your runtime:

- **T4 (16 GB):** use `--dtype float16`. Llama-3-8B weights alone are ~16 GB so loading is tight and OOM at long context is expected — those failure rows are the data point that shows the memory wall.
- **A100 (40 GB):** `bfloat16` is fine for the full 8k→64k sweep.

If you skipped the HF login above, override the model with `--model-id` to something non-gated (e.g. `Qwen/Qwen2.5-1.5B-Instruct`, `microsoft/phi-2`).

In [13]:
# T4 example (Llama-3-8B in fp16)
# !python scripts/run_context_sweep.py \
#   --config configs/baseline_hf.yaml \
#   --dtype float16 \
#   --context-lengths 2048 4096 8192 16384 \
#   --max-new-tokens 64

# A100 example — uncomment if applicable
# !python scripts/run_context_sweep.py \
#   --config configs/baseline_hf.yaml \
#   --context-lengths 8192 16384 32768 65536 \
#   --max-new-tokens 128

!python scripts/run_context_sweep.py \
  --config configs/baseline_hf.yaml \
  --model-id Qwen/Qwen2.5-1.5B-Instruct \
  --dtype float16 \
  --context-lengths 1024 2048 4096 8192 16384 32768 \
  --max-new-tokens 64 \
  --results-path results/phase1.jsonl

Loading Qwen/Qwen2.5-1.5B-Instruct (dtype=float16) on cuda ...
config.json: 100% 660/660 [00:00<00:00, 3.14MB/s]
tokenizer_config.json: 7.30kB [00:00, 11.2MB/s]
vocab.json: 2.78MB [00:00, 10.8MB/s]
merges.txt: 1.67MB [00:00, 8.56MB/s]
tokenizer.json: 7.03MB [00:00, 17.1MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors: 100% 3.09G/3.09G [00:28<00:00, 110MB/s]
Loading weights: 100% 338/338 [00:07<00:00, 43.03it/s, Materializing param=model.norm.weight]
generation_config.json: 100% 242/242 [00:00<00:00, 1.26MB/s]
Model loaded.

--- context_length=1024, batch_size=1 ---
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
  success=True ttft=1.

## 7. Summarize

In [14]:
!python scripts/summarize_results.py --results-dir results/


=== Overview ===
Total runs:    9
Successes:     6
Failures:      3
Models:        ['Qwen/Qwen2.5-1.5B-Instruct', 'sshleifer/tiny-gpt2']
Backends:      ['huggingface']
Hardware:      ['cuda:Tesla T4']

=== Failures (3) ===
                model_name     backend      hardware  context_length  batch_size                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               error
       sshleifer/tiny-gpt2 huggingface cuda:Tesla T4            1024           1                                                                   

## 8. Save results before the runtime disconnects

Free Colab resets `/content` on disconnect. Either download to your laptop or push back to GitHub.

In [15]:
# Option A: download to laptop
from google.colab import files
files.download("results/baseline_hf.jsonl")

# Option B: push to GitHub from inside Colab (uncomment, edit name/email)
# !git config user.email "you@example.com"
# !git config user.name "Your Name"
# !git add results/ && git commit -m "Colab T4 baseline run" && git push

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>